# Практика. Агент на LangChain

**Эталонный ноутбук для преподавателя.** Полностью рабочий: заполнить нужно
только `base_url` и `model` в ячейке с моделью.

Код-аналог агента из n8n: то же самое, но видно устройство.
В конце — знакомство с MCP на готовом сервере.

| В n8n | Здесь |
|---|---|
| нода Chat Model | `llm = ChatOpenAI(...)` |
| нода-инструмент + поле Description | функция с `@tool` и docstring |
| поле System Message | строка `SYSTEM_PROMPT` |
| нода Simple Memory | `InMemorySaver()` + `thread_id` |
| нода AI Agent | `create_agent(...)` |
| Executions → Logs | `show_calls(...)` |

## Шаг 1. Окружение и модель

In [ ]:
%pip install -q "langchain>=1.0,<2" "langgraph>=1.0,<2" langchain-openai

In [ ]:
from importlib.metadata import version

for p in ("langchain", "langgraph", "langchain-openai"):
    print(p, version(p))

In [ ]:
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

# Ключ только через getpass: вписанный в ячейку, он уедет вместе с ноутбуком.
if "LLM_API_KEY" not in os.environ:
    os.environ["LLM_API_KEY"] = getpass("Введите API-ключ: ")

llm = ChatOpenAI(
    base_url="ЗАПОЛНИТЬ: эндпоинт провайдера",
    api_key=os.environ["LLM_API_KEY"],
    model="ЗАПОЛНИТЬ: модель с поддержкой вызова инструментов",
    temperature=0,          # агенту нужна дисциплина, а не креативность
)

print(llm.invoke("Привет! Ответь одним словом.").content)

## Шаг 2. Инструменты — и что из них видит модель

Ядро занятия. Инструмент — обычная функция; для агента она превращается
в имя + описание + схему аргументов.

In [ ]:
from langchain.tools import tool

NOTES: list[str] = []


@tool(parse_docstring=True)
def save_note(text: str) -> str:
    """Сохраняет заметку пользователя, чтобы вернуться к ней позже.

    Используй, когда пользователь просит что-то запомнить, записать или отметить.
    Не используй для обычных вопросов и для чтения уже сохранённого.

    Args:
        text: текст заметки одной фразой.
    """
    NOTES.append(text)
    return f"Сохранено. Всего заметок: {len(NOTES)}"


@tool(parse_docstring=True)
def list_notes() -> list[str]:
    """Возвращает все сохранённые заметки пользователя.

    Используй, когда пользователь спрашивает, что было записано ранее.
    Не используй, когда пользователь просит что-то запомнить.
    """
    return NOTES or ["Заметок пока нет."]

### Ячейка, ради которой стоит занятие

Модель **не видит тело функции**. Она видит ровно то, что напечатано ниже.
Отсюда: docstring — часть программы, а не комментарий.

In [ ]:
TOOLS = [save_note, list_notes]

for t in TOOLS:
    print("ИМЯ:       ", t.name)
    print("ОПИСАНИЕ:  ", (t.description or "").replace("\n", " ")[:160])
    print("АРГУМЕНТЫ: ", t.args)
    print("-" * 70)

### Мини-эксперимент

Без `parse_docstring=True` описания аргументов в схему не попадают —
остаются только имена и типы. Сравните вывод.

In [ ]:
@tool                      # тот же docstring, но БЕЗ parse_docstring
def save_note_plain(text: str) -> str:
    """Сохраняет заметку пользователя.

    Args:
        text: текст заметки одной фразой.
    """
    return "ok"


print("С parse_docstring: ", save_note.args)
print("Без parse_docstring:", save_note_plain.args)

## Шаг 3. Агент

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

SYSTEM_PROMPT = """
Ты — ассистент первокурсника по учебным вопросам.

Правила:
1. Вызывай инструмент ТОЛЬКО если он нужен для ответа. На обычные вопросы отвечай текстом.
2. Если для вызова не хватает данных — не выдумывай их, а спроси у пользователя.
3. Не вызывай один и тот же инструмент дважды за один ответ.
4. Отвечай по-русски, не длиннее 5 предложений.
"""

# Одна строка = нода AI Agent из n8n. Внутреннюю логику написали за нас.
agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

## Шаг 4. Память

Память не «внутри агента» — она привязана к `thread_id`.
В n8n ровно это делало поле Session ID у Simple Memory.

In [ ]:
def ask(text: str, thread_id: str = "student-1"):
    return agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config={"configurable": {"thread_id": thread_id}, "recursion_limit": 15},
    )


print(ask("Привет, я Аня")["messages"][-1].content)
print(ask("Как меня зовут?")["messages"][-1].content)
print("--- другой thread_id ---")
print(ask("Как меня зовут?", thread_id="student-2")["messages"][-1].content)

## Шаг 5. Лог вызовов и три проверки

`show_calls` — здешний аналог Executions → Logs. Текст ответа доказательством
не является: агент может написать «записал», ничего не вызвав.

In [ ]:
def show_calls(result):
    for m in result["messages"]:
        for call in getattr(m, "tool_calls", []) or []:
            print(f"ВЫЗОВ:     {call['name']}({call['args']})")
        if m.__class__.__name__ == "ToolMessage":
            print(f"РЕЗУЛЬТАТ: {str(m.content)[:200]}")
    print("ОТВЕТ:    ", result["messages"][-1].content)

In [ ]:
# Проверка 1: обычный вопрос → ни одного вызова
show_calls(ask("Что такое академическая задолженность?", thread_id="check"))

In [ ]:
# Проверка 2: просьба запомнить → вызван save_note
show_calls(ask("Запиши, что мне тяжело даётся матанализ", thread_id="check"))

In [ ]:
# Проверка 3: вопрос о записанном → вызван list_notes
# (в новом thread_id память пуста, поэтому достать заметку можно только вызовом)
show_calls(ask("А что я просил записать?", thread_id="fresh"))

## Шаг 6. MCP: инструменты живут на другом сервере

Своего сервера не пишем — берём готовый `budget_server.py`.
Он скачивается прямо в среду, загружать файл руками не нужно.

Версия `mcp` зафиксирована ниже 2.0: с 2.x не работает MCP-клиент LangChain.

In [ ]:
%pip install -q langchain-mcp-adapters "mcp>=1.9,<2" nest_asyncio

In [ ]:
# В Colab обязательно: иначе asyncio.run() конфликтует с циклом ноутбука.
import nest_asyncio
nest_asyncio.apply()

In [ ]:
SERVER_URL = ("https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/"
              "AI%20Culture/QualifiedCustomer/lesson05/lesson05_budget_server.py")

!wget -q -O budget_server.py "{SERVER_URL}"
!fuser -k 8000/tcp 2>/dev/null || true
!nohup python budget_server.py > mcp_server.log 2>&1 &

In [ ]:
import time
time.sleep(5)
!tail -n 6 mcp_server.log          # ждём строку Uvicorn running on http://0.0.0.0:8000

### Главное свойство MCP

Мы не импортировали ни одной функции. В ноутбуке нет ни `calculate_budget`,
ни `add_saving`. Но агент знает их имена, назначение и аргументы —
всё пришло по сети из другого процесса.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "server": {"url": "http://localhost:8000/mcp/", "transport": "streamable_http"}
})

mcp_tools = await client.get_tools()

print(f"Обнаружено инструментов: {len(mcp_tools)}\n")
for t in mcp_tools:
    print("ИМЯ:       ", t.name)
    print("ОПИСАНИЕ:  ", (t.description or "").replace("\n", " ")[:120])
    print("АРГУМЕНТЫ: ", list(t.args))
    print("-" * 70)

### Тот же агент, но на чужих инструментах

Код не меняется — меняется только список `tools`.

In [ ]:
mcp_agent = create_agent(
    model=llm,
    tools=mcp_tools,
    system_prompt="Все расчёты делай только через инструменты. Не считай в уме.",
    checkpointer=InMemorySaver(),
)

r = await mcp_agent.ainvoke(
    {"messages": [{"role": "user", "content":
        "Доход 30000, аренда 12000, еда 9000, транспорт 2000. "
        "Хочу откладывать 5000. Посчитай и скажи, что урезать."}]},
    config={"configurable": {"thread_id": "demo"}, "recursion_limit": 15},
)
show_calls(r)

## Шаг 7. Обсуждение

- Где здесь ваш код, а где чужой? Что написано за вас в `create_agent`?
- Что видит модель вместо тела функции? Почему функция без docstring невидима?
- Чем `thread_id` отличается от «памяти внутри агента»?
- Описания MCP-инструментов писали не вы. Что будет, если автор сервера их поменяет?
- Мы собрали того же агента, что и в n8n. Когда какой способ уместнее?
- В каких задачах агент **не нужен** и хватит обычного конвейера?

---

### Дальше — лабораторная

- **Уровень 4** — шаги 1–5 с вашей темой и вашими инструментами.
- **Уровень 5** — шаг 6, доведённый до конца: своя пара инструментов на сервере
  и доказательство, что состояние живёт там, а не в ноутбуке.

Шаблоны с заготовками ячеек — в `Лабораторная/Материалы`.